In [1]:
import geass
import pickle
import pandas as pd
import networkx as nx
from rdkit import Chem
from rdkit.Chem.PandasTools import LoadSDF
from constants import REACTIONS, REACTANTS, FILLING_ATOMS, BROKEN_BONDS, BROKEN_BONDS_LIPIDS

print('Networkx version :', nx.__version__)

Networkx version : 2.2


# 1. Loading molecules libraries

In [2]:
# Load the BUL (Biological unit library) or Natural products library
with open('../recap-delivery/data/BUL_names.pkl', 'rb') as f:
    BULnames = pickle.load(f)

In [3]:
# Load the BUL (Biological unit library) or Natural products library
# with open('../recap-delivery/data/BUL_names.pkl', 'rb') as f:
with open('data/BUL_with_lipids.pkl', 'rb') as f:
    BULnames_lipids = pickle.load(f)

In [4]:
# Loads the PBU

with open('metadisassembler5/data/PBU_v2.pkl', 'rb') as f:
    PBU = pickle.load(f)

In [5]:
# Loads the BUL

with open('metadisassembler5/data/BUL_Mar23.pkl', 'rb') as f:
    BUL = pickle.load(f)

In [6]:
# Load the ACTUAL SECRETed 1853 molecules
sdf_filename = 'data/SECRETed_space_Feb23.sdf'
df_complete = LoadSDF(sdf_filename, smilesName='SMILES', molColName='Molecule', strictParsing=True)
df_smiles = df_complete[['ID','SMILES', 'Molecule']]
df_aux_ids = pd.read_csv('data/id_smiles_Feb23.csv', index_col=0)
df_smiles['IDs'] = df_aux_ids['IDs']

/home/ubuntu/anaconda3/envs/map4_test2/lib/python3.7/site-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [7]:
df_complete = pd.read_csv('data/Database3_no_SS.csv')

In [8]:
df_complete['Molecule'] = df_complete.apply(lambda x: Chem.MolFromSmiles(x['Isomeric_SMILES']), axis=1)

In [9]:
df_complete.rename({'Isomeric_SMILES':'SMILES'}, axis=1, inplace=True)
df_complete.rename({'ID_unique':'IDs'}, axis=1, inplace=True)
df_smiles = df_complete[['IDs','SMILES', 'Molecule']]

In [10]:
# Assign the SECRETed molecules to the collection
mols = list(df_smiles['Molecule'])
# This indexer will help us to control the molecules
indexer_original_mols = list(df_smiles['IDs'])

## 1.1. Remove the modifications from the NP database

In [11]:
# The modifications in the NP database are not needed.

print('Size before revoming: ', len(BULnames))
BULnames_copy = {k:v for k, v in BULnames.items()}
for names, mol in BULnames.items():
    if len(names) == 1:
        if 'mod' in names[0]:
            del BULnames_copy[names]
            
BULnames = {k:v for k, v in BULnames_copy.items()}       
print('Size after revoming: ', len(BULnames))

Size before revoming:  20035
Size after revoming:  18409


In [47]:
fragmentator = geass.Geass(mols[indexer_original_mols.index('SF_0406')], BROKEN_BONDS, REACTANTS, REACTIONS, FILLING_ATOMS, BULnames, verbose=0) # Default Geass

In [48]:
fragmentator.get_bonds_to_break()

In [49]:
fragmentator.create_fragmentation_graph()

In [50]:
fragmentator.plot_graph()

CytoscapeWidget(cytoscape_layout={'name': 'circle', 'nodeSpacing': 200, 'edgeLengthVal': 100}, cytoscape_style…

# 2. Run the process with all the compounds

## 2.1. Create graphs

#### <span style="color:red">BEWARE</span>: Aprox time to execute with 1853 molecules: 15-20 mins

#### <span style="color:red">BEWARE</span>: Aprox time to execute with 4034 molecules: 60 mins aprox

In [19]:
result_graphs, disassembled_ids, analytics_result = geass.create_graphs(mols, indexer_original_mols, PBU, BUL, BROKEN_BONDS, REACTANTS, REACTIONS, FILLING_ATOMS, BULnames)

PROGRESS: [4034/4034]
Elapsed: 3580.301s


In [53]:
result_graphs, disassembled_ids, analytics_result = geass.create_graphs([mols[indexer_original_mols.index('SF_0406')]], indexer_original_mols, PBU, BUL, BROKEN_BONDS, REACTANTS, REACTIONS, FILLING_ATOMS, BULnames)

PROGRESS: [1/1]


Our lib correctly loaded! Of size: 16652
Our PBU correctly loaded! Of size: 5006
No boundary bonds
Our lib correctly loaded! Of size: 16652
Our PBU correctly loaded! Of size: 5006
No boundary bonds
Our lib correctly loaded! Of size: 16652
Our PBU correctly loaded! Of size: 5006
No boundary bonds
Error
Our lib correctly loaded! Of size: 16652
Our PBU correctly loaded! Of size: 5006
Len seed_list before optimising: 1
Finished.
Our lib correctly loaded! Of size: 16652
Our PBU correctly loaded! Of size: 5006
Len seed_list before optimising: 12
Finished.
Elapsed: 835.371s


#### <span style="color:red">BEWARE</span>: Aprox time to execute with 4034 molecules: 90 mins aprox

In [28]:
result_graphs_lipids, disassembled_ids_lipids, analytics_result_lipids = geass.create_graphs(mols, indexer_original_mols, PBU, BUL, BROKEN_BONDS, REACTANTS, REACTIONS, FILLING_ATOMS, BULnames_lipids)

PROGRESS: [4034/4034]
Elapsed: 5499.719s


In [ ]:
# result_graphs, disassembled_ids, analytics_result = geass.create_graphs(mols, indexer_original_mols, PBU, BUL, BROKEN_BONDS_LIPIDS, REACTANTS, REACTIONS, FILLING_ATOMS, BULnames)

### 2.1.1. Optional: Store analytics

In [25]:
csv_string = 'ID_UNIQUE,"Compounds ids"\n'
for k, v in disassembled_ids.items():
    csv_string += k + ',"' + str(v) + '"\n'
with open('dissasembled_Dec23.csv', 'w') as csv_file:
    csv_file.write(csv_string)

In [29]:
csv_string = 'ID_UNIQUE,"Compounds ids"\n'
for k, v in disassembled_ids_lipids.items():
    csv_string += k + ',"' + str(v) + '"\n'
with open('dissasembled_lipids_Dec23.csv', 'w') as csv_file:
    csv_file.write(csv_string)

In [26]:
analytics_df = pd.DataFrame()
analytics_df['Fragmented All NP'] = analytics_result['Fragmented All NP']
analytics_df = pd.concat([analytics_df, pd.DataFrame({'Fragmented not NP OK': analytics_result['Fragmented not NP OK']})], axis=1)
analytics_df = pd.concat([analytics_df, pd.DataFrame({'Fragmented not NP ERROR': analytics_result['Fragmented not NP ERROR']})], axis=1)
analytics_df = pd.concat([analytics_df, pd.DataFrame({'Not fragmented All NP': analytics_result['Not fragmented All NP']})], axis=1)
analytics_df = pd.concat([analytics_df, pd.DataFrame({'Not fragmented not NP OK': analytics_result['Not fragmented not NP OK']})], axis=1)
analytics_df = pd.concat([analytics_df, pd.DataFrame({'Not fragmented not NP ERROR': analytics_result['Not fragmented not NP ERROR']})], axis=1)

In [30]:
analytics_lipids_df = pd.DataFrame()
analytics_lipids_df['Fragmented All NP'] = analytics_result_lipids['Fragmented All NP']
analytics_lipids_df = pd.concat([analytics_lipids_df, pd.DataFrame({'Fragmented not NP OK': analytics_result_lipids['Fragmented not NP OK']})], axis=1)
analytics_lipids_df = pd.concat([analytics_lipids_df, pd.DataFrame({'Fragmented not NP ERROR': analytics_result_lipids['Fragmented not NP ERROR']})], axis=1)
analytics_lipids_df = pd.concat([analytics_lipids_df, pd.DataFrame({'Not fragmented All NP': analytics_result_lipids['Not fragmented All NP']})], axis=1)
analytics_lipids_df = pd.concat([analytics_lipids_df, pd.DataFrame({'Not fragmented not NP OK': analytics_result_lipids['Not fragmented not NP OK']})], axis=1)
analytics_lipids_df = pd.concat([analytics_lipids_df, pd.DataFrame({'Not fragmented not NP ERROR': analytics_result_lipids['Not fragmented not NP ERROR']})], axis=1)

In [27]:
analytics_df.to_csv("geass_and_MD_analytics_Dec23.csv", index=False)

In [31]:
analytics_lipids_df.to_csv("geass_and_MD_analytics_lipids_Dec23.csv", index=False)

## 2.2. Create compound edges collection for similarity calculation

In [20]:
compound_edges = geass.create_compound_edges(result_graphs)

In [32]:
compound_edges_lipids = geass.create_compound_edges(result_graphs_lipids)

## 2.3. Create similarity matrix

In [21]:
similarity_df = geass.create_similarity_matrix(compound_edges)

Elapsed: 415.564s


In [33]:
similarity_lipids_df = geass.create_similarity_matrix(compound_edges_lipids)

Elapsed: 404.359s


### 2.3.1. Optional: Store similarity matrix

In [22]:
similarity_df.to_csv('similarity_matrix_Dec23.csv')

In [34]:
similarity_lipids_df.to_csv('similarity_matrix_lipids_Dec23.csv')

# 3. Same process but with MIBiG

## 3.1. Loading required molecules

In [174]:
sdf_filename = 'data/MIBIG-3.1-mergedComps.sdf'
df_mibig = LoadSDF(sdf_filename, smilesName='SMILES', molColName='Molecule', strictParsing=True)
# Molecules
mibig_mols = list(df_mibig['Molecule'])
# Indexes
mibig_indexes = list(df_mibig['ID'])

RDKit WARNING: [14:04:02] WARNING: not removing hydrogen atom without neighbors
RDKit WARNING: [14:04:02] WARNING: not removing hydrogen atom without neighbors
RDKit WARNING: [14:04:02] WARNING: not removing hydrogen atom without neighbors
RDKit WARNING: [14:04:02] WARNING: not removing hydrogen atom without neighbors
RDKit ERROR: [11:50:33] ERROR: moving to the beginning of the next molecule
RDKit ERROR: [14:04:03] ERROR: Counts line too short: '' on line474511


## 3.2. Run Geass + MD

#### <span style="color:red">BEWARE</span>: Aprox time to execute with MIBiG molecules: 25 mins

In [179]:
result_mibig_graphs, disassembled_mibig_ids, analytics_result_mibig = geass.create_graphs(mibig_mols, mibig_indexes, PBU, BUL, BROKEN_BONDS, REACTANTS, FILLING_ATOMS, BULnames)

PROGRESS: [14/8]
SMILES: CC
Our lib correctly loaded! Of size: 16652
Our PBU correctly loaded! Of size: 5006
Len seed_list before optimising: 1
Finished.
Elapsed: 1521.256s


## 3.3. Create compound edges collection

In [196]:
compound_edges_mibig = geass.create_compound_edges(result_mibig_graphs)

### 3.3.1. Combine SECRETed compound edges with MIBiG compound edges

In [200]:
compound_edges_all = compound_edges.copy()
compound_edges_all.update(compound_edges_mibig)

## 3.4. Get similarity matrix

In [205]:
similarity_df_all = geass.create_similarity_matrix(compound_edges_all)

Elapsed: 241.475s


### 3.4.1. Optional: store similarity matrix

In [207]:
# similarity_df_all.to_csv('similarity_matrix_mibig.csv')